# VCC 2026 · 3a — top 1,000 calls (leaderboard probe)

Same model, Δ tables and generator as `vcc_2026_3` (submission 3, overall +0.091); the only change is **N = 1,000 calls per knockdown instead of 2,000** (detection floor at z = 5 kept).
Why: submission 3's Jaccard dipped (0.026 → −0.024) while H1 rewarded the extra calls, so the challenge lines may have fewer real DE genes than H1. H1 itself preferred 2,000 (scaled overall 0.118 vs 0.104), so this submission tests whether H1 is a good guide for N on A–C.
Nothing is refit or re-validated: the settings are those of the `model2_top1000_floor` arm that `vcc_2026_3`'s finalists run already scored on H1, and the code is `vcc_2026_3`'s (`~/vcc/code/vcc_2026_3`).

## 0. Session
Paths, the `run` / `bg` / `status` helpers, and input checks. `WORK3` is `vcc_2026_3`'s folder (read only); this notebook writes to `WORK` and `LOGS`.

In [ ]:
import os, sys, shutil, subprocess, shlex, json, copy
from pathlib import Path
import pandas as pd
V = Path.home() / 'vcc'
ENV = dict(CODE=str(V / 'code' / 'vcc_2026_3'), DATA=str(V / 'data'), LIB=str(V / 'lib'), LOGS=str(V / 'logs' / 'vcc_2026_3a'),
           B='gs://vcc2026', PY=sys.executable, WORK=str(V / 'lib' / 'vcc_2026_3a'), WORK3=str(V / 'lib' / 'vcc_2026_3'))
os.environ.update(ENV)
for d in ('LOGS', 'WORK'):
    os.makedirs(ENV[d], exist_ok=True)

def run(cmd):
    """Short step: run now (from the code folder), stream the output here, stop the cell on error."""
    p = subprocess.Popen(['bash', '-c', 'set -eo pipefail; cd "$CODE"\n' + cmd], env=os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}) - see the output above')

def bg(name, cmd):
    """Long step: run in tmux session `name`; output goes to ~/vcc/logs/vcc_2026_3a/<name>.log."""
    session = 'vcc3a_' + name
    if subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0:
        raise RuntimeError(f'{session} is already running; wait for completion')
    log, sh = Path(ENV['LOGS']) / f'{name}.log', Path(ENV['LOGS']) / f'{name}.sh'
    exports = ''.join(f'export {k}={shlex.quote(v)}\n' for k, v in ENV.items())
    sh.write_text(f'#!/bin/bash\n{exports}exec > {shlex.quote(str(log))} 2>&1\n'
                  f'trap \'echo "EXIT=$?"\' EXIT\nset -eo pipefail\ncd "$CODE"\n{cmd}\n')
    subprocess.run(['tmux', 'new-session', '-d', '-s', session, 'bash', str(sh)], check=True)
    print(f'started "{name}" in the background. Check with: status("{name}")')

def status(name, n=20):
    session = 'vcc3a_' + name
    alive = subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0
    log = Path(ENV['LOGS']) / f'{name}.log'
    lines = log.read_text().splitlines() if log.exists() else []
    state = 'RUNNING' if alive else ('FINISHED OK' if lines and lines[-1] == 'EXIT=0' else ('NOT STARTED' if not lines else 'STOPPED - read the log below'))
    print(f'== {name}: {state}\n' + '\n'.join(lines[-n:]))

# ---- checks ----
FROM = Path(ENV['WORK3']) / 'val_final'      # vcc_2026_3's finalists run: settings and H1 scores
LINES = ['A', 'B', 'C']                      # use D/E/F with the final bundle
need = [Path(ENV['CODE']) / 'vcc3_generate.py', FROM / 'selected_generator.json', FROM / 'scaled_scores.csv',
        V / 'data/gene_names.csv', V / 'data/pert_counts.csv'] + [Path(ENV['WORK3']) / 'pred' / line for line in LINES]
missing = [str(f) for f in need if not f.exists()]
print('python:', sys.executable)
print('inputs: all present' if not missing else f'MISSING inputs: {missing} (run vcc_2026_3 sections 4 and 6 first)')
print('tmux:', 'OK' if shutil.which('tmux') else 'MISSING -> in the terminal: sudo apt-get install -y tmux')
TOP_N = 1000
RUN_NAME = f'vcc_2026_3a_top{TOP_N}'

## 1. Settings
`vcc_2026_3`'s selected settings with N taken from the scored `model2_top{N}_floor` arm. Stops unless N is the only setting that differs, and shows both arms' H1 scores (leaderboard formula).

In [ ]:
base = json.loads((FROM / 'selected_generator.json').read_text())
ARM = f'model2_top{TOP_N}_floor'
arm = json.loads((FROM / f'{ARM}.json').read_text())
scaled = pd.read_csv(FROM / 'scaled_scores.csv', index_col='arm')
if (arm['method'], arm['call'], arm.get('cell_rule', 'fixed')) != (base['delta_array'], base['settings']['call'], base['cell_rule']):
    raise ValueError(f'{ARM} is not a variant of the selected arm {base["arm"]}')
chosen = copy.deepcopy(base)
for key in ('top_n', 'detect_floor', 'z_req', 'alpha'):
    chosen['settings'][key] = arm[key]
changed = {k: (base['settings'][k], v) for k, v in chosen['settings'].items() if base['settings'][k] != v}
if set(changed) != {'top_n'}: raise ValueError(f'expected only top_n to change, got {changed}')
chosen.update(arm=ARM, scaled_overall=float(scaled.loc[ARM, 'overall']),
              margin=float(scaled.loc[ARM, 'overall'] - chosen['margin_over_overall']),
              selection=f'leaderboard probe: {ARM} from vcc_2026_3 val_final (H1 selected {base["arm"]})')
SETTINGS = Path(ENV['WORK']) / 'generator_settings.json'
SETTINGS.write_text(json.dumps(chosen, indent=1) + '\n')
print('changed:', changed)
display(scaled.loc[[base['arm'], ARM]].round(4))

## 2. Generate
Cells for every challenge line from `vcc_2026_3`'s Δ tables with the settings above (needs ~20 GB free disk; about 50 minutes); refuses to overwrite a finished run, so use a new `RUN_NAME` each time.

In [ ]:
result = V / 'results' / RUN_NAME
if result.exists() and any(result.iterdir()): raise FileExistsError('Choose a new RUN_NAME to preserve existing results.')
free = shutil.disk_usage(V).free / 1e9
if free < 20: raise RuntimeError(f'only {free:.0f} GB free; a submission needs ~20 GB')
line_args = ' '.join(shlex.quote(line) for line in LINES)
delta_args = ' '.join(f'--delta {shlex.quote(line + "=" + str(Path(ENV["WORK3"]) / "pred" / line))}' for line in LINES)
bg('generate', f'"$PY" vcc3_generate.py submit --data "$DATA" --lines {line_args} --delta-array selected --settings {shlex.quote(str(SETTINGS))} {delta_args} --out {shlex.quote(str(result))}')

In [ ]:
status('generate')

## 3. Package
`vcc prep` validates and packages the run. It holds the whole matrix in memory (~30 GB at peak), so it stops early unless RAM + swap is enough; leaderboard submission and bucket backup stay off until switched on.

In [ ]:
SUBMISSION = V / 'results' / RUN_NAME
memory = {k: int(v.split()[0]) / 1e6 for k, v in (line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines()) if k in ('MemAvailable', 'SwapFree')}
if memory['MemAvailable'] + memory['SwapFree'] < 34:
    raise RuntimeError(f"only {memory['MemAvailable']:.0f} GB RAM + {memory['SwapFree']:.0f} GB swap free; vcc prep peaks near 30 GB. Add swap first.")
run(f'vcc prep {shlex.quote(str(SUBMISSION / "prediction.h5ad"))} -g "$DATA/gene_names.csv" --perts "$DATA/pert_counts.csv" -o {shlex.quote(str(SUBMISSION / "prediction.vcc"))}')

In [ ]:
SUBMIT_TO_LEADERBOARD = True
MODEL_NAME = 'LHY'   # shown publicly on the leaderboard: keep it free of method details
if SUBMIT_TO_LEADERBOARD:
    s = json.loads(SETTINGS.read_text())
    print(f"submitting as {MODEL_NAME!r}; private record: {s['arm']} ({s['delta_array']}), top_n={s['settings']['top_n']}, "
          f"alpha={s['settings']['alpha']}, validation scaled overall={s['scaled_overall']:.4f}")
    run(f'vcc submit {shlex.quote(str(SUBMISSION / "prediction.vcc"))} -m {shlex.quote(MODEL_NAME)} --wait')

In [ ]:
BACK_UP_TO_BUCKET = False
if BACK_UP_TO_BUCKET:
    run(r"""
    gcloud storage rsync -r "$WORK" "$B/lib/vcc_2026_3a"
    gcloud storage cp ~/vcc/vcc_2026_3a.ipynb "$B/code/vcc_2026_3a.ipynb"
    """)